In [ ]:
'''
MODEL CONTEXT PROTOCOL (MCP) + MACHINE LEARNING EXAMPLE
This project teaches:

Machine Learning: Data cleaning, Feature encoding, Classification
Model evaluation: Python, Pandas, Scikit-learn,
Functions: API-style development
MCP: MCP Server, MCP Tools, Tool Registration, Agent Integration, AI Tool Calling

* This code was writen for Google Colab use.
Dataset: Adult Income:  https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data

NOTE:
     Running a full MCP server in Colab often causes asyncio errors.
     For learning purposes, this example creates and tests MCP tools without starting the server.

What is the Model Context Protocol (MCP)?
     The Model Context Protocol (MCP) is an open standard that enables artificial intelligence models,
     applications, and agents to securely connect to external tools, data sources, databases, APIs, and
     business systems. It provides a common communication framework that allows AI systems to discover
     available tools, request information, and perform actions without requiring custom integrations for
     every application.

     Before MCP, developers typically had to create a unique integration for each AI model and each external
     service. For example, connecting an AI assistant to a database, a CRM system, a weather API, and a file
     storage service would require separate code for each connection. MCP standardizes these interactions so
     that AI applications can communicate with many different tools using the same protocol.

What Does MCP Do?
     MCP acts as a bridge between AI models and external resources. It allows an AI model to:
- Access databases and retrieve information.
- Read and write files.
- Query APIs and web services.
- Interact with business applications.
- Execute functions and tools.
- Access enterprise knowledge bases.
- Retrieve real-time information.
- Automate workflows and business processes.

Instead of relying solely on information contained within the model's training data, MCP enables
the AI to obtain current, relevant, and organization-specific information when needed.

MCP Architecture
MCP typically consists of three components:
1. MCP Client
  The application or AI agent that requests information or actions.
  Examples:
  - AI chatbots
  - Virtual assistants
  - Agentic AI systems
  - Custom AI applications
2. MCP Server
  A service that exposes tools, resources, and functions to MCP clients.
  Examples:
  - Python MCP servers
  - Database servers
  - File system servers
  - API wrapper servers
3. Resources and Tools
  The actual functionality made available through the MCP server.
  Examples:
  - Database queries
  - Machine learning predictions
  - Customer record lookups
  - Report generation
  - Data analysis functions
'''


In [1]:
# INSTALL REQUIRED PACKAGES

!pip install -q pandas scikit-learn mcp


In [2]:
# IMPORT LIBRARIES

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from mcp.server.fastmcp import FastMCP

print ("Libraries ready")

Libraries ready


In [3]:
# LOAD DATASET

dataset_url = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/adult/adult.data"
)

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv(
    dataset_url,
    names=columns,
    skipinitialspace=True
)

print("Dataset Loaded")
print(df.head())


Dataset Loaded
   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=50K  
1             0            

In [4]:
# PREPROCESS DATA

df = df.dropna()

encoders = {}

for col in df.columns:
    if df[col].dtype == object:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col])
        encoders[col] = encoder

X = df.drop("income", axis=1)
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
print("Data processing")

Data processing


In [5]:
# TRAIN MACHINE LEARNING MODEL

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")



Model Accuracy: 86.17 %


In [6]:
# CREATE MCP SERVER

mcp = FastMCP("Income Predictor MCP Server")

print("\nMCP Server Created")


MCP Server Created


In [7]:
# CREATE MCP TOOL

@mcp.tool()
def predict_income(
    age: int,
    education_num: int,
    hours_per_week: int,
    capital_gain: int
) -> str:
    """
    Predict whether income exceeds $50K.
    """

    sample = [[
        age,                 # age
        0,                   # workclass
        0,                   # fnlwgt
        0,                   # education
        education_num,       # education_num
        0,                   # marital_status
        0,                   # occupation
        0,                   # relationship
        0,                   # race
        0,                   # sex
        capital_gain,        # capital_gain
        0,                   # capital_loss
        hours_per_week,      # hours_per_week
        0                    # native_country
    ]]

    prediction = model.predict(sample)[0]

    return f"Predicted Income Class: {prediction}"


print("MCP Tool Registered")


MCP Tool Registered


In [8]:
# TEST MCP TOOL DIRECTLY

result = predict_income(
    age=45,
    education_num=13,
    hours_per_week=40,
    capital_gain=5000
)

print("\nPrediction Result:")
print(result)



Prediction Result:
Predicted Income Class: 0


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [9]:
# LIST AVAILABLE MCP TOOLS

print("\nAvailable MCP Tool:")
print(" - predict_income")


Available MCP Tool:
 - predict_income


In [ ]:
# OPTIONAL SERVER STARTUP
#
# DO NOT RUN THIS IN GOOGLE COLAB
#
# Uncomment and run on a local machine
# (VS Code, PyCharm, Windows Terminal, Linux Terminal)
#
# if __name__ == "__main__":
#     mcp.run()
#

In [10]:
print("\nMCP Learning Example Completed Successfully")


MCP Learning Example Completed Successfully
